# ET Agent — Evolving Treatment Analysis

In [ ]:
import pathlib
import sys
import os

import anthropic
import pandas as pd
from IPython.display import display, Markdown

_here = pathlib.Path().resolve()
if (_here / 'clinical-trial-optimizer').exists():
    CT_ROOT = _here / 'clinical-trial-optimizer'
elif (_here / 'src').exists():
    CT_ROOT = _here
elif (_here.parent / 'src').exists():
    CT_ROOT = _here.parent
else:
    raise RuntimeError(f'Cannot locate clinical-trial-optimizer root from {_here}')

sys.path.insert(0, str(CT_ROOT / 'src'))

from agent_prompt_functions import get_et_agent_prompt
from data_utils import filterETLOT, formatETLotDF

print(f'CT_ROOT : {CT_ROOT}')

In [ ]:
print('API key set' if os.environ.get('ANTHROPIC_API_KEY') else 'WARNING: ANTHROPIC_API_KEY not set')

In [ ]:
DATA_FILE = CT_ROOT / 'data' / 'rwd_lot.xlsx'

sheets = pd.read_excel(DATA_FILE, sheet_name=None)
print(f'Sheets found: {list(sheets.keys())}')

# Adjust sheet names to match your workbook
period1_raw = sheets['period1']
period2_raw = sheets['period2']

PERIOD1_RANGE = '2017-2020'
PERIOD2_RANGE = '2021-2024'

et_raw = filterETLOT(period1_raw, period2_raw)
et_df = formatETLotDF(et_raw)

print(f'Rows after filterETLOT: {len(et_raw)}')
print(et_df[:500])

In [ ]:
# Paste I/E criteria here
ie_criteria = ""


In [ ]:
et_prompt = get_et_agent_prompt(ie_criteria, et_df, period1_range=PERIOD1_RANGE, period2_range=PERIOD2_RANGE)
print(f'Prompt length: {len(et_prompt)} chars')

In [ ]:
# ⚠️ HUMAN APPROVAL REQUIRED — this cell calls the Claude API
client = anthropic.Anthropic()

response = client.messages.create(
    model='claude-haiku-4-5-20251001',
    max_tokens=4096,
    messages=[{'role': 'user', 'content': et_prompt}],
)

display(Markdown(response.content[0].text))